In [57]:
import pandas as pd
import numpy as np
import plotly.express as px
from collections import defaultdict
import warnings
import os
warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)

def build_dp_table(mrc_dict, max_total_slabs, trace_names, access_freqs, pretty_print=False):
    """
    Builds the DP table and allocation table for the given trace names and maximum total slabs.

    Parameters:
    mrc_dict (dict): A nested dictionary that maps trace_name to their miss ratio at different slab_cnt.
    max_total_slabs (int): The maximum number of slabs to consider.
    trace_names (list): The trace names that we are interested in.
    access_freqs (list): The access frequencies for the trace names.
    pretty_print (bool): If True, pretty print the DP table and allocation table.

    Returns:
    tuple: (dp, allocation) where:
        - dp: The DP table storing the minimum weighted miss ratio for each trace and slab count.
        - allocation: The allocation table storing the number of slabs allocated to each trace.
    """
    # Number of traces
    n = len(trace_names)
    
    # Initialize the DP table
    dp = [[float('inf')] * (max_total_slabs + 1) for _ in range(n + 1)]
    dp[0][0] = 0  # Base case: 0 slabs for 0 traces has a miss ratio of 0
    
    # Initialize the allocation table
    allocation = [[0] * (max_total_slabs + 1) for _ in range(n + 1)]
    
    # Fill the DP table
    for i in range(1, n + 1):
        trace_name = trace_names[i - 1]
        access_freq = access_freqs[i - 1]
        for j in range(max_total_slabs + 1):
            for k in range(j + 1):
                miss_ratio = mrc_dict[trace_name].get(k, 1)
                miss_count = miss_ratio * access_freq
                if dp[i - 1][j - k] + miss_count < dp[i][j]:
                    dp[i][j] = dp[i - 1][j - k] + miss_count
                    allocation[i][j] = k
    
    # Pretty print the DP table and allocation table if requested
    if pretty_print:
        print("DP Table:")
        for row in dp:
            print(', '.join([f'{x:.4f}' for x in row]))
        print("\nAllocation Table:")
        for row in allocation:
            print(', '.join([f'{x:3d}' for x in row]))
    
    return dp, allocation


def backtrack_allocation(dp, allocation, trace_names, total_slabs, access_freqs):
    """
    Performs backtracking on the precomputed DP table to determine the optimal allocation for a given total_slabs.

    Parameters:
    dp (list): The DP table built by `build_dp_table`.
    allocation (list): The allocation table built by `build_dp_table`.
    trace_names (list): The trace names that we are interested in.
    total_slabs (int): The total number of slabs to allocate.
    access_freqs (list): The access frequencies for the trace names.

    Returns:
    tuple: (result, normalized_miss_ratio) where:
        - result: A dictionary with the optimal allocation of slabs for each trace name.
        - normalized_miss_ratio: The minimized weighted miss ratio normalized by the total access frequency.
    """
    # Number of traces
    n = len(trace_names)
    
    # Backtrack to find the optimal allocation
    result = {}
    j = total_slabs
    for i in range(n, 0, -1):
        trace_name = trace_names[i - 1]
        result[trace_name] = allocation[i][j]
        j -= allocation[i][j]
    
    # Normalized miss ratio
    normalized_miss_ratio = dp[n][total_slabs] / sum(access_freqs)
    
    return result, normalized_miss_ratio



def compute_optimal_allocations(mrc_dict, mrc_delta_dict, wss_slabs_dict, max_total_slabs, trace_names, access_freqs):
    """
    Compute the optimal slab allocations and miss ratios for each total_slab from 1 to max_total_slabs.

    Parameters:
    mrc_dict (dict): A nested dictionary that maps trace_name to their miss ratio at different slab_cnt.
    max_total_slabs (int): The maximum number of slabs to consider.
    trace_names (list): The trace names that we are interested in.
    access_freqs (list): The access frequencies for the trace names.

    Returns:
    pd.DataFrame: A DataFrame where each row corresponds to a total_slab and contains:
        - Columns for each trace_name (number of slabs allocated to the trace).
        - 'total_miss_ratio': The normalized miss ratio for the given total_slab.
        - 'total_slab_cnt': The total number of slabs.
    """

    dp, allocation = build_dp_table(mrc_dict, max_total_slabs, trace_names, access_freqs)


    results = []
    for total_slab in range(1, max_total_slabs + 1):
        alloc, miss_ratio = backtrack_allocation(dp, allocation, trace_names, total_slab, access_freqs)
        # no more increase after it saturates
        row = {trace_name: min(alloc[trace_name], wss_slabs_dict[trace_name]) for trace_name in trace_names}
        row['total_miss_ratio'] = miss_ratio
        row['total_slab_cnt'] = total_slab
        for trace_name in trace_names:
            row[f"{trace_name}_miss_ratio"] = mrc_dict[trace_name][alloc[trace_name]]
            row[f"{trace_name}_miss_ratio_delta"] = mrc_delta_dict[trace_name][alloc[trace_name]]
        results.append(row)

    results_df = pd.DataFrame(results)
    return results_df

In [58]:
import heapq
import pandas as pd

def greedy_allocation_with_snapshots(mrc_dict, mrc_delta_dict, wss_slabs_dict, max_total_slabs, trace_names, access_freqs):
    """
    Greedy approach to allocate slabs based on utility, with tracking of allocation order and snapshots.

    Parameters:
    mrc_dict (dict): A nested dictionary that maps trace_name to their miss ratio at different slab counts.
    mrc_delta_dict (dict): A nested dictionary that maps trace_name to the reduction in miss ratio (delta) for each additional slab.
    max_total_slabs (int): The maximum number of slabs to allocate.
    trace_names (list): The trace names (class names) to allocate slabs to.
    access_freqs (list): The access frequencies for each trace.

    Returns:
    tuple: (allocation, normalized_miss_ratio, allocation_order, snapshots_df) where:
        - allocation: A dictionary mapping each trace_name to the number of slabs allocated.
        - normalized_miss_ratio: The normalized miss ratio after all slabs are allocated.
        - allocation_order: A list tracking the order in which slabs were allocated to traces.
        - snapshots_df: A DataFrame where each row corresponds to a snapshot of the allocation at a given total_slab.
    """

    allocation = {trace_name: 0 for trace_name in trace_names}
    allocation_order = []  
    snapshots = []  

    max_heap = []
    for i, trace_name in enumerate(trace_names):
        utility = mrc_delta_dict[trace_name][1] * access_freqs[i]
        heapq.heappush(max_heap, (-utility, False, i, trace_name))


    for total_slab in range(1, max_total_slabs + 1):
        if not max_heap:
            break  

        neg_utility, index, saturated, trace_name = heapq.heappop(max_heap)
        current_slabs = allocation[trace_name]
        allocation[trace_name] += 1  
        allocation_order.append(trace_name)  

        next_slabs = current_slabs + 1
        if next_slabs + 1 in mrc_delta_dict[trace_name]:  
            next_utility = mrc_delta_dict[trace_name][next_slabs + 1] * access_freqs[index]
            # Push (-utility, index, trace_name) to the heap to maintain tie-breaking
            heapq.heappush(max_heap, (-next_utility, index, next_slabs >= wss_slabs_dict[trace_name], trace_name))
        # no more increase after it saturates
        snapshot = {trace_name: min(allocation[trace_name], wss_slabs_dict[trace_name]) for trace_name in trace_names}
        snapshot['total_slab_cnt'] = total_slab
        snapshot['total_miss_ratio'] = sum(
            mrc_dict[trace_name][allocation[trace_name]] * access_freqs[i]
            for i, trace_name in enumerate(trace_names)
        ) / sum(access_freqs)
        for trace_name in trace_names:
            snapshot[f"{trace_name}_miss_ratio"] = mrc_dict[trace_name][allocation[trace_name]]
            snapshot[f"{trace_name}_miss_ratio_delta"] = (
                mrc_delta_dict[trace_name][allocation[trace_name]]
                if allocation[trace_name] in mrc_delta_dict[trace_name]
                else 0
            )
        snapshots.append(snapshot)

    # Calculate the normalized miss ratio
    total_miss_ratio = 0
    total_access_freq = sum(access_freqs)
    for i, trace_name in enumerate(trace_names):
        slabs_allocated = allocation[trace_name]
        miss_ratio = mrc_dict[trace_name][slabs_allocated]
        total_miss_ratio += miss_ratio * access_freqs[i]

    normalized_miss_ratio = total_miss_ratio / total_access_freq

    # Convert snapshots to a DataFrame
    snapshots_df = pd.DataFrame(snapshots)

    return allocation, normalized_miss_ratio, allocation_order, snapshots_df

In [59]:
import re
def process_chunked_subtraces(directory):
    
    chunk_dirs = []
    pattern = re.compile(r'^chunk_\d+$')  # Regex to match 'chunk_xxx' where xxx are digits

    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)
        if os.path.isdir(subdir_path) and pattern.match(subdir):
            chunk_dirs.append(subdir)
    if not chunk_dirs:
        chunk_dirs = [directory]
    
    optimal_dp_miss_ratios = []
    optimal_dp_allocations = {}
    optimal_greedy_miss_ratios = []
    optimal_greedy_allocations = {}
    
    for chunk_dir in chunk_dirs:
        chunk_path = os.path.join(directory, chunk_dir)
        miss_ratios_path = os.path.join(chunk_path, "miss_ratios.csv")
        subtrace_stat_path = os.path.join(chunk_path, "subtrace_stat.csv")
        
        subtrace_miss_ratio_df = pd.read_csv(miss_ratios_path)
        subtrace_stat_df = pd.read_csv(subtrace_stat_path) 

       
        subtrace_miss_ratio_df['class_size'] = subtrace_miss_ratio_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
        subtrace_stat_df['class_size'] = subtrace_stat_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
        subtrace_stat_df['wss_slabs'] = np.ceil((subtrace_stat_df['distinct_object_count'] * subtrace_stat_df['class_size']) / (4 * 1024 * 1024)).astype(int)
        
        
        records = subtrace_miss_ratio_df.to_dict(orient='records')
        mrc_dict = defaultdict(dict)
        mrc_delta_dict = defaultdict(dict)

        for record in records:
            mrc_dict[record['class_size']][record['slab_cnt']] = record['miss_ratio']
            mrc_delta_dict[record['class_size']][record['slab_cnt']] = record['miss_ratio_delta']
            

        for class_size in mrc_dict:
            mrc_dict[class_size][0] = 1
            mrc_delta_dict[class_size][0] = float('inf')

        wss_slabs_dict = {}
        for record in subtrace_stat_df.to_dict(orient='records'):
            wss_slabs_dict[record['class_size']] = record['wss_slabs']

        class_sizes = sorted(mrc_dict.keys())
        
        access_freqs = {
            r['class_size']: r['record_count']
            for r in subtrace_stat_df.to_dict(orient='records')
        }

        slab_upper_limit = 1024
        optim_allocs_df = compute_optimal_allocations(mrc_dict, mrc_delta_dict, wss_slabs_dict, slab_upper_limit, list(mrc_dict.keys()), [access_freqs[k] for k in mrc_dict.keys()])
        _, _, greedy_order, greedy_snapshots_df = greedy_allocation_with_snapshots(
            mrc_dict, mrc_delta_dict, wss_slabs_dict, slab_upper_limit, sorted(list(mrc_dict.keys()), reverse=True), [access_freqs[k] for k in sorted(list(mrc_dict.keys()), reverse=True)]
        )

        total_records_cnt = sum(access_freqs.values())
        optimal_dp_miss_ratios.append((total_records_cnt, {r['total_slab_cnt']: r['total_miss_ratio'] for r in optim_allocs_df.to_dict(orient='records')}))
        for r in optim_allocs_df.to_dict(orient='records'):
            optimal_dp_allocations[(chunk_dir, r['total_slab_cnt'])] = {k: v for k, v in r.items() if k in class_sizes}
        optimal_greedy_miss_ratios.append((total_records_cnt, {r['total_slab_cnt']: r['total_miss_ratio'] for r in greedy_snapshots_df.to_dict(orient='records')}))
        for r in greedy_snapshots_df.to_dict(orient='records'):
            optimal_greedy_allocations[(chunk_dir, r['total_slab_cnt'])] = {k: v for k, v in r.items() if k in class_sizes}
    def compute_weighted_averages(miss_ratios):
        weighted_averages = defaultdict(float)
        total_records_per_slab = defaultdict(int)

        for total_records_cnt, miss_ratio_dict in miss_ratios:
            for total_slab, miss_ratio in miss_ratio_dict.items():
                weighted_averages[total_slab] += total_records_cnt * miss_ratio
                total_records_per_slab[total_slab] += total_records_cnt

        for total_slab in weighted_averages:
            weighted_averages[total_slab] /= total_records_per_slab[total_slab]

        return dict(weighted_averages)

    weighted_avg_dp = compute_weighted_averages(optimal_dp_miss_ratios)
    weighted_avg_greedy = compute_weighted_averages(optimal_greedy_miss_ratios)

    return weighted_avg_dp, weighted_avg_greedy, optimal_dp_allocations, optimal_greedy_allocations, greedy_order
    

In [60]:

trace_name = 'synth_static_202'


simulation_path = "report.csv"
simulation_df = pd.read_csv(simulation_path)
simulation_df = simulation_df[simulation_df['trace_name'] == trace_name]
simulation_df['slab_cnt'] = (simulation_df['cacheSizeMB'] - 4) // 4


base_dir = f"/mydata/hongshu/traces/thesis/subtraces/{trace_name}/chunk_0"
miss_ratios_path = os.path.join(base_dir, "miss_ratios.csv")
subtrace_stat_path = os.path.join(base_dir, "subtrace_stat.csv")

subtrace_miss_ratio_df = pd.read_csv(miss_ratios_path) if os.path.exists(miss_ratios_path) else None
subtrace_stat_df = pd.read_csv(subtrace_stat_path) if os.path.exists(subtrace_stat_path) else None


subtrace_miss_ratio_df['class_size'] = subtrace_miss_ratio_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
subtrace_stat_df['class_size'] = subtrace_stat_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
subtrace_stat_df['wss_slabs'] = np.ceil((subtrace_stat_df['distinct_object_count'] * subtrace_stat_df['class_size']) / (4 * 1024 * 1024)).astype(int)
subtrace_stat_df['wss'] = (subtrace_stat_df['distinct_object_count'] * subtrace_stat_df['class_size'])


optimal_lookup_dict, greedy_optimal_lookup_dict, optimal_dp_allocations, optimal_greedy_allocations, greedy_order\
    = process_chunked_subtraces(f"/mydata/hongshu/traces/thesis/subtraces/{trace_name}/")

optimal_allocs = []
for slab, mr in optimal_lookup_dict.items():
    alloc = optimal_dp_allocations.get(('chunk_0', slab), {})
    alloc['total_slab_cnt'] = slab
    alloc['total_miss_ratio'] = mr
    optimal_allocs.append(alloc)
optimal_allocs_df = pd.DataFrame(optimal_allocs)

greedy_allocs = []
for slab, mr in greedy_optimal_lookup_dict.items():
    alloc = optimal_greedy_allocations.get(('chunk_0', slab), {})
    alloc['total_slab_cnt'] = slab
    alloc['total_miss_ratio'] = mr
    greedy_allocs.append(alloc)
greedy_allocs_df = pd.DataFrame(greedy_allocs)


optimal_allocation_slabs_lookup = {}
for record in optimal_allocs_df.to_dict(orient='records'):
    optimal_allocation_slabs_lookup[record['total_slab_cnt']] = {
        256: record[256], 
        512: record[512],
        1024: record[1024],
        2048: record[2048], 
        4096: record[4096]
    }

In [15]:
optimal_allocs_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_optimal_allocs.csv", index=False)

In [5]:
optimal_allocs_df[optimal_allocs_df['total_slab_cnt'].isin([32, 64, 128, 256])]

,1024,2048,256,512,4096,total_slab_cnt,total_miss_ratio
31,0,12,0,12,8,32,0.624007
63,15,15,10,14,10,64,0.551321
127,48,17,36,16,11,128,0.416731
255,97,28,89,26,16,256,0.190462


In [61]:
import ast

def parse_acStats(acStats_str):
    try:
        parsed_list = ast.literal_eval(acStats_str)
        return {entry['allocSize']: entry['totalSlabs'] for entry in parsed_list}
    except Exception as e:
        print(f"Error parsing: {acStats_str}, Error: {e}")
        return {}


analysis_df = simulation_df[simulation_df['trace_name'] == trace_name]
analysis_df['class_slabs'] = analysis_df['_acStats'].apply(parse_acStats)


In [7]:
analysis_df.columns

Index(['directory', 'allocator', 'lruRefreshSec', 'cacheSizeMB',
       'moveOnSlabRelease', 'anomalyDetectionFrequency', 'rebalanceStrategy',
       'poolRebalanceIntervalSec', 'tailSlabCnt', 'wakeUpRebalancerEveryXReqs',
       'mhMovingAverageParam', 'allocSizes', 'trace_name', 'cache_size',
       'uuid', 'extra', 'cache_sizes', 'file_path', '_rebalancerNumRuns',
       '_ramEvictions', '_getMissRatio', '_deltaStats', '_poolUsageFraction',
       '_rebalancerAvgRebalanceTimeMs', '_evicAttempts', '_allocFailures',
       '_poolUnusedFraction', '_poolUsableSize', '_effectiveMovementRates',
       '_anomalyCount', '_rebalanceEvents', '_perPoolFreeMemorySize',
       '_perPoolFragmentationSize', '_rebalancerAvgPickTimeMs',
       '_rebalancerPickVictimRounds', '_allocAttempts', '_nvmItem',
       '_poolFragementationFraction', '_missRatios', '_rebalanceReqIds',
       '_getCnt', '_getMissCnt', '_totalMissCnt', '_acEvictionAgeStats',
       '_acStats', '_rebalancerAvgReleaseTimeMs',
   

In [13]:
analysis_df = analysis_df[analysis_df['slab_cnt'].isin([128, 256])]
"""
rebalance_strategy, slab_cnt, rebalance_interval, miss_ratio, rebalanced_slabs

rebalance_strategy, slab_cnt, rebalance_interval, miss_ratio_over_time
"""
all_interval_results = []
for record in analysis_df.to_dict(orient='records'):
    slab_cnt = record['slab_cnt']
    rebalanced_slabs = record['_rebalancerNumRebalancedSlabs']
    miss_ratio = record['_missRatio']
    rebalance_interval = record['wakeUpRebalancerEveryXReqs']
    rebalanced_strategy = record['rebalanceStrategy']
    all_interval_results.append({
        'rebalance_strategy': rebalanced_strategy,
        'slab_cnt': slab_cnt,
        'rebalance_interval': rebalance_interval,
        'miss_ratio': miss_ratio,
        'rebalanced_slabs': rebalanced_slabs
    })
all_interval_results.append({
    'rebalance_strategy': 'optimal',
    'slab_cnt': 128,
    'rebalance_interval': 0,
    'miss_ratio': optimal_lookup_dict[128],
    'rebalanced_slabs': 0
})
all_interval_results.append({
    'rebalance_strategy': 'optimal',
    'slab_cnt': 256,
    'rebalance_interval': 0,
    'miss_ratio': optimal_lookup_dict[256],
    'rebalanced_slabs': 0
})
all_interval_df = pd.DataFrame(all_interval_results)
    
    

In [19]:
all_interval_df[(all_interval_df['rebalance_strategy'] == 'optimal') & (all_interval_df['slab_cnt'] == 256)].sort_values(by='miss_ratio')

,rebalance_strategy,slab_cnt,rebalance_interval,miss_ratio,rebalanced_slabs
813,optimal,256,0,0.190462,0


In [18]:
all_interval_df[(all_interval_df['rebalance_strategy'] == 'marginal-hits') & (all_interval_df['slab_cnt'] == 256)].sort_values(by='miss_ratio')

,rebalance_strategy,slab_cnt,rebalance_interval,miss_ratio,rebalanced_slabs
645,marginal-hits,256,120000,0.189965,642
170,marginal-hits,256,150000,0.189990,514
204,marginal-hits,256,110000,0.189992,701
502,marginal-hits,256,130000,0.190067,593
732,marginal-hits,256,100000,0.190178,770
...,...,...,...,...,...
333,marginal-hits,256,960000,0.220727,80
88,marginal-hits,256,970000,0.221133,79
8,marginal-hits,256,980000,0.221630,78
397,marginal-hits,256,990000,0.222012,77


In [14]:
all_interval_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_all_interval_results.csv", index=False)

In [24]:
"""
fixed interval: 100_000
figure group 1: 
miss ratio vs. slab_count [32, 64, 128, 256], group by strategies, 
we should also include the optimal allocation

figure group 2:
distance to optimal allocation vs. slab_count [32, 64, 128, 256], group by strategies,

schema: 
rebalance_strategy, slab_count, miss_ratio, number of rebalanced slabs, distance to optimal allocation.

"""
fixed_interval_plotting_data = []
for record in analysis_df[(analysis_df['wakeUpRebalancerEveryXReqs'] == 100_000) | (analysis_df['rebalanceStrategy'] == 'disabled')].to_dict(orient='records'):
    class_slabs = record['class_slabs']
    slab_cnt = record['slab_cnt']
    optimal_alloc = optimal_allocation_slabs_lookup[slab_cnt]
    distance_to_optimal_allocation = sum(
        abs(class_slabs.get(size, 0) - optimal_alloc.get(size, 0)) for size in optimal_alloc
    ) // 2
    fixed_interval_plotting_data.append({
        'rebalance_strategy': record['rebalanceStrategy'],
        'slab_count': record['slab_cnt'],
        'miss_ratio': record['_missRatio'],
        'number_of_rebalanced_slabs': record['_rebalancerNumRebalancedSlabs'],
        'distance_to_optimal_allocation': distance_to_optimal_allocation
    })
for slab_cnt in [32, 64, 128, 256]:
    fixed_interval_plotting_data.append({
        'rebalance_strategy': 'optimal',
        'slab_count': slab_cnt,
        'miss_ratio': optimal_lookup_dict[slab_cnt],
        'number_of_rebalanced_slabs': 0,
        'distance_to_optimal_allocation': 0
    })
fixed_interval_plotting_data_df = pd.DataFrame(fixed_interval_plotting_data)
    

In [7]:
analysis_df["_missRatios"].values.tolist()[0]

"{'51300000': {'reqDelta': 100000, 'missDelta': 26994, 'missRatio': 0.26994}, '46200000': {'reqDelta': 100000, 'missDelta': 26910, 'missRatio': 0.2691}, '40500000': {'reqDelta': 100000, 'missDelta': 26887, 'missRatio': 0.26887}, '33300000': {'reqDelta': 100000, 'missDelta': 26949, 'missRatio': 0.26949}, '31400000': {'reqDelta': 100000, 'missDelta': 26776, 'missRatio': 0.26776}, '3800000': {'reqDelta': 100000, 'missDelta': 26718, 'missRatio': 0.26718}, '37000000': {'reqDelta': 100000, 'missDelta': 26918, 'missRatio': 0.26918}, '0': {'reqDelta': 0, 'missDelta': 0, 'missRatio': 0}, '400000': {'reqDelta': 100000, 'missDelta': 57264, 'missRatio': 0.57264}, '3500000': {'reqDelta': 100000, 'missDelta': 26974, 'missRatio': 0.26974}, '4900000': {'reqDelta': 100000, 'missDelta': 26581, 'missRatio': 0.26581}, '54600000': {'reqDelta': 100000, 'missDelta': 26941, 'missRatio': 0.26941}, '73100000': {'reqDelta': 100000, 'missDelta': 26744, 'missRatio': 0.26744}, '75300000': {'reqDelta': 100000, 'miss

In [ ]:
import ast
"""
miss ratios over time
schema:
rebalance_strategy, slab_count, miss_ratio, request_id, we also need a line where all slabs ha
"""
miss_ratio_over_time_data = []
for record in analysis_df.to_dict(orient='records'):
    miss_ratio_details = ast.literal_eval(record['_missRatios'])
    slab_cnt = record['slab_cnt']
    for k, v in miss_ratio_details.items():
        request_id = int(k)
        mr = v['missRatio']
        miss_ratio_over_time_data.append({
            'rebalance_strategy': record['rebalanceStrategy'],
            'slab_count': slab_cnt,
            'miss_ratio': mr,
            'request_id': request_id
        })


In [ ]:
miss_ratio_over_time_data_df = pd.DataFrame(miss_ratio_over_time_data)
miss_ratio_over_time_data_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_miss_ratio_over_time.csv", index=False)

In [11]:
miss_ratio_over_time_data_df

,rebalance_strategy,slab_count,miss_ratio,request_id,distance_to_optimal_allocation
0,disabled,256,0.26994,51300000,85
1,disabled,256,0.26910,46200000,85
2,disabled,256,0.26887,40500000,85
3,disabled,256,0.26949,33300000,85
4,disabled,256,0.26776,31400000,85
...,...,...,...,...,...
15995,free-mem,256,0.27092,62900000,85
15996,free-mem,256,0.26923,63700000,85
15997,free-mem,256,0.26933,47800000,85
15998,free-mem,256,0.26602,42900000,85


In [25]:
fixed_interval_plotting_data_df.sort_values(by=['slab_count', 'rebalance_strategy'])

,rebalance_strategy,slab_count,miss_ratio,number_of_rebalanced_slabs,distance_to_optimal_allocation
8,disabled,32,0.640658,0,12
2,free-mem,32,0.640658,0,12
13,hits,32,0.627721,7,6
15,marginal-hits,32,0.627866,797,3
20,optimal,32,0.624007,0,0
16,tail-age,32,0.640658,0,12
11,disabled,64,0.564820,0,13
18,free-mem,64,0.564820,0,13
4,hits,64,0.565739,19,23
1,marginal-hits,64,0.554192,795,3


In [26]:
fixed_interval_plotting_data_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_fixed_interval_plotting_data.csv", index=False)

In [25]:
compare_df_records = []
for record in analysis_df.to_dict(orient='records'):
    disabled_distribution = [record['class_slabs'][2048], record['class_slabs'][4096]]
    optimal_distribution = [optimal_allocation_slabs_lookup[record['slab_cnt']][2048], optimal_allocation_slabs_lookup[record['slab_cnt']][4096]]
    compare_df_records.append({
        'total_slabs': record['slab_cnt'],
        'miss_ratio_disabled': record['_missRatio'],
        'miss_ratio_optimal': optimal_lookup_dict[record['slab_cnt']],
        'slab_distribution_disabled': disabled_distribution,
        'slab_distribution_optimal': optimal_distribution,
        'distance_to_optimal': (abs(disabled_distribution[0] - optimal_distribution[0]) + abs(disabled_distribution[1] + optimal_distribution[1])) // 2,
    })  
    
compare_df = pd.DataFrame(compare_df_records)

In [27]:
compare_df

,total_slabs,miss_ratio_disabled,miss_ratio_optimal,slab_distribution_disabled,slab_distribution_optimal,distance_to_optimal
0,15,0.393512,0.388615,"[5, 10]","[8, 7]",10
1,413,0.104223,0.100735,"[138, 275]","[188, 225]",275
2,406,0.105610,0.102087,"[136, 270]","[183, 223]",270
3,145,0.196516,0.191428,"[49, 96]","[72, 73]",96
4,509,0.087580,0.085028,"[170, 339]","[218, 291]",339
...,...,...,...,...,...,...
511,299,0.131988,0.127518,"[100, 199]","[144, 155]",199
512,119,0.214297,0.209122,"[40, 79]","[60, 59]",79
513,147,0.195594,0.190200,"[49, 98]","[73, 74]",98
514,449,0.097394,0.094276,"[150, 299]","[199, 250]",299


In [26]:
compare_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/default_allocation/{trace_name}_compare.csv", index=False)

In [6]:
analysis_df.sort_values(by='slab_cnt', inplace=True)
analysis_df[['slab_cnt', 'class_slabs', '_missRatio']]

,slab_cnt,class_slabs,_missRatio
785,2,"{2048: 1, 4096: 1}",0.553556
482,3,"{2048: 1, 4096: 2}",0.525790
164,4,"{2048: 2, 4096: 2}",0.497580
672,5,"{2048: 2, 4096: 3}",0.481120
583,6,"{2048: 2, 4096: 4}",0.469355
...,...,...,...
9,509,"{2048: 170, 4096: 339}",0.087580
673,510,"{2048: 170, 4096: 340}",0.087469
114,511,"{2048: 171, 4096: 340}",0.087246
274,512,"{2048: 171, 4096: 341}",0.087137


In [8]:
def compare_with_optimal(analysis_df, optimal_allocs_df, slab_cnt):
    row = analysis_df[analysis_df['slab_cnt'] == slab_cnt]
    if row.empty:
        raise ValueError(f"No row found in analysis_df for slab_cnt={slab_cnt}")
    row = row.iloc[0]
    class_slabs = row['class_slabs']  # dict like {2048: 21, 4096: 43}
    miss_ratio = row['_missRatio']

    # Find the corresponding row in optimal_allocs_df
    opt_row = optimal_allocs_df[optimal_allocs_df['total_slab_cnt'] == slab_cnt]
    if opt_row.empty:
        raise ValueError(f"No row found in optimal_allocs_df for total_slab_cnt={slab_cnt}")
    opt_row = opt_row.iloc[0]
    optimal_miss_ratio = opt_row['total_miss_ratio']

    # Compute miss ratio diff
    miss_ratio_diff = miss_ratio - optimal_miss_ratio

    # Compute allocation distance (Manhattan distance / 2)
    alloc_distance = 0
    for k in class_slabs:
        alloc_distance += abs(class_slabs.get(k, 0) - opt_row.get(k, 0))
    alloc_distance /= 2

    return {
        'miss_ratio': float(miss_ratio),
        'optimal_miss_ratio': float(optimal_miss_ratio),
        'miss_ratio_diff': float(miss_ratio_diff),
        'miss_ratio_diff_pct': float(miss_ratio_diff / optimal_miss_ratio) if optimal_miss_ratio != 0 else float('inf'),
        'alloc_distance': int(alloc_distance),
        'class_slabs': class_slabs,
        'optimal_alloc': {k: float(opt_row.get(k, 0)) for k in class_slabs}
    }

In [26]:
compare_with_optimal(analysis_df, optimal_allocs_df, 64)

{'miss_ratio': 0.62043275,
 'optimal_miss_ratio': 0.5948908,
 'miss_ratio_diff': 0.025541949999999924,
 'miss_ratio_diff_pct': 0.04293552699083583,
 'alloc_distance': 26,
 'class_slabs': {2048: 11, 4096: 53},
 'optimal_alloc': {2048: 37.0, 4096: 27.0}}

In [9]:
import json

with open(f"outcome2/synth_thesis_static_103_disabled_128_5000/out.json") as f:
    data = json.load(f)


In [10]:
slab_over_time = [{0:0, 1:0, 'request_id': 0}]
for item in data['snapshots']:
    slab_over_time.append({
        0: item['numSlabs']['0'],
        1: item['numSlabs']['1'],
        'request_id': item['request_id']
    })
slab_over_time_df = pd.DataFrame(slab_over_time)

In [11]:
slab_over_time_df.to_csv("/mydata/hongshu/thesis-playground/thesis-plotting/scripts/default_allocation/synth_thesis_static_103_disabled_128_slab_over_time.csv", index=False)

In [81]:
slab_over_time_df

,0,1,request_id
0,0,0,0
1,1,1,1000
2,1,1,2000
3,1,1,3000
4,1,2,4000
...,...,...,...
19995,43,85,19995000
19996,43,85,19996000
19997,43,85,19997000
19998,43,85,19998000


In [90]:
t_name = 'synth_thesis_static_100'
base_dir = f"/mydata/hongshu/traces/thesis/subtraces/{t_name}/chunk_0"
miss_ratios_path = os.path.join(base_dir, "miss_ratios.csv")
subtrace_stat_path = os.path.join(base_dir, "subtrace_stat.csv")
alloc_size_path = os.path.join(f"/mydata/hongshu/traces/thesis/subtraces/{t_name}/", "alloc_size.json")

miss_ratio_df = pd.read_csv(miss_ratios_path) 
miss_ratio_df['class_size'] = miss_ratio_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
subtrace_stat_df = pd.read_csv(subtrace_stat_path)
with open(alloc_size_path, 'r') as f:
    alloc_sizes = json.load(f)



In [93]:
miss_ratio_df[miss_ratio_df['slab_cnt'] == 256]

,subtrace_name,cache_size,slab_cnt,miss_count,miss_ratio,miss_ratio_delta,class_size
255,synth_thesis_static_100_subtrace_2048.csv,1073741824,256,625975,0.062598,0.000150,2048
1279,synth_thesis_static_100_subtrace_4096.csv,1073741824,256,1103519,0.110352,0.000329,4096


In [94]:
subtrace_stat_df

,subtrace_name,record_count,distinct_object_count,zipf_slope,zipf_intercept,zipf_r2,zipf_p_value
0,synth_thesis_static_100_subtrace_2048.csv,10000000,598574,-1.063798,14.248078,0.971051,0.0
1,synth_thesis_static_100_subtrace_4096.csv,10000000,598500,-1.064490,14.255227,0.971185,0.0


In [20]:
import os
import json
strategy_name = "tail-age"
total_slabs = 256
input_dir = f"outcome_detail/synth_static_202_{strategy_name}_{total_slabs}"

with open(os.path.join(input_dir, "out.json"), 'r') as f:
    data = json.load(f)

data['snapshots'][0]


{'allSlabsAllocated': False,
 'request_id': 100000,
 'freeMemory': {'4': 0, '3': 0, '2': 0, '1': 0, '0': 0},
 'numSlabs': {'4': 8, '3': 7, '2': 5, '1': 2, '0': 2},
 'missEstimation': {'4': 12351, '3': 7637, '2': 540, '1': 5599, '0': 141},
 'hitsPerSlab': {'4': 1543, '3': 1091, '2': 108, '1': 2799, '0': 70},
 'evictions': {'4': 0, '3': 0, '2': 0, '1': 0, '0': 0},
 'hits': {'4': 12351, '3': 7637, '2': 540, '1': 5599, '0': 141},
 'marginalHits': {'4': 690, '3': 834, '2': 178, '1': 1930, '0': 110},
 'tailAge': {'4': 756, '3': 756, '2': 756, '1': 756, '0': 756}}

In [12]:
import os

def list_subdirs(directory):
    return [name for name in os.listdir(directory)
            if os.path.isdir(os.path.join(directory, name))]


In [18]:
"""
{'allSlabsAllocated': False,
 'request_id': 100000,
 'freeMemory': {'4': 0, '3': 0, '2': 0, '1': 0, '0': 0},
 'numSlabs': {'4': 8, '3': 7, '2': 5, '1': 2, '0': 2},
 'missEstimation': {'4': 12351, '3': 7637, '2': 540, '1': 5599, '0': 141},
 'hitsPerSlab': {'4': 1543, '3': 1091, '2': 108, '1': 2799, '0': 70},
 'evictions': {'4': 0, '3': 0, '2': 0, '1': 0, '0': 0},
 'hits': {'4': 12351, '3': 7637, '2': 540, '1': 5599, '0': 141},
 'marginalHits': {'4': 690, '3': 834, '2': 178, '1': 1930, '0': 110},
 'tailAge': {'4': 756, '3': 756, '2': 756, '1': 756, '0': 756}}
"""
import json

detail_subdirs = list_subdirs("outcome_detail")
distance_to_optimal = []
for subdir in detail_subdirs:
    with open(os.path.join("outcome_detail", subdir, "out.json"), 'r') as f:
        data = json.load(f)
    with open(os.path.join("outcome_detail", subdir, "config.json"), 'r') as f:
        config = json.load(f)
    with open(os.path.join("outcome_detail", subdir, "stats.json"), 'r') as f:
        stats = json.load(f)

    """
    'numSlabs': {'4': 8, '3': 7, '2': 5, '1': 2, '0': 2},
    """
    sizes = [256, 512, 1024, 2048, 4096]
    slab_cnt = (config["cache_config"]["cacheSizeMB"] - 4) // 4
    rebalance_strategy = config["cache_config"]["rebalanceStrategy"]
    for snap in data['snapshots']:
        class_slabs = {sizes[int(k)]: v for k, v in snap['numSlabs'].items()}
        optimal_alloc = optimal_allocation_slabs_lookup[slab_cnt]
        distance_to_optimal_allocation = sum(
            abs(class_slabs.get(size, 0) - optimal_alloc.get(size, 0)) for size in optimal_alloc
        ) // 2
        distance_to_optimal.append({
            'rebalance_strategy': rebalance_strategy,
            'slab_count': slab_cnt,
            'request_id': snap['request_id'],
            'distance_to_optimal_allocation': distance_to_optimal_allocation
        })
distance_to_optimal_df = pd.DataFrame(distance_to_optimal)
distance_to_optimal_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_distance_to_optimal_over_time.csv", index=False)
    
    

In [ ]:
optimal_allocation_slabs_lookup[256]



{256: 89, 512: 26, 1024: 97, 2048: 28, 4096: 16}

In [22]:
analysis_df[(analysis_df['rebalanceStrategy'] == "disabled") & (analysis_df['slab_cnt'] == 256)][['class_slabs', '_missRatio']]

,class_slabs,_missRatio
38,"{256: 29, 512: 28, 1024: 72, 2048: 63, 4096: 64}",0.273442
95,"{256: 29, 512: 28, 1024: 72, 2048: 63, 4096: 64}",0.273442


In [43]:
analysis_df.columns

Index(['directory', 'allocator', 'lruRefreshSec', 'cacheSizeMB',
       'moveOnSlabRelease', 'anomalyDetectionFrequency', 'rebalanceStrategy',
       'poolRebalanceIntervalSec', 'tailSlabCnt', 'wakeUpRebalancerEveryXReqs',
       'mhMovingAverageParam', 'allocSizes', 'trace_name', 'cache_size',
       'uuid', 'extra', 'cache_sizes', 'file_path', '_rebalancerNumRuns',
       '_ramEvictions', '_getMissRatio', '_deltaStats', '_poolUsageFraction',
       '_rebalancerAvgRebalanceTimeMs', '_evicAttempts', '_allocFailures',
       '_poolUnusedFraction', '_poolUsableSize', '_effectiveMovementRates',
       '_anomalyCount', '_rebalanceEvents', '_perPoolFreeMemorySize',
       '_perPoolFragmentationSize', '_rebalancerAvgPickTimeMs',
       '_rebalancerPickVictimRounds', '_allocAttempts', '_nvmItem',
       '_poolFragementationFraction', '_missRatios', '_rebalanceReqIds',
       '_getCnt', '_getMissCnt', '_totalMissCnt', '_acEvictionAgeStats',
       '_acStats', '_rebalancerAvgReleaseTimeMs',
   

In [62]:
"""
subdirs = [
    "synth_static_202_marginal-hits_256_1000",
    "synth_static_202_marginal-hits_256_10000",
    "synth_static_202_marginal-hits_256",
    "synth_static_202_marginal-hits_256_500000",
    "synth_static_202_marginal-hits_256_1000000"
]
"""
target_df = analysis_df[((analysis_df['rebalanceStrategy'] == "disabled") & (analysis_df['slab_cnt'] == 256)) | 
                        (
                        (analysis_df['rebalanceStrategy'] == "marginal-hits") & (analysis_df['slab_cnt'] == 256) & 
                        analysis_df['wakeUpRebalancerEveryXReqs'].isin([1000, 10000, 100000, 500000, 1000000])
                        )]

In [48]:
target_df['_missRatios'].values.tolist()[0]

"{'51300000': {'reqDelta': 100000, 'missDelta': 26994, 'missRatio': 0.26994}, '46200000': {'reqDelta': 100000, 'missDelta': 26910, 'missRatio': 0.2691}, '40500000': {'reqDelta': 100000, 'missDelta': 26887, 'missRatio': 0.26887}, '33300000': {'reqDelta': 100000, 'missDelta': 26949, 'missRatio': 0.26949}, '31400000': {'reqDelta': 100000, 'missDelta': 26776, 'missRatio': 0.26776}, '3800000': {'reqDelta': 100000, 'missDelta': 26718, 'missRatio': 0.26718}, '37000000': {'reqDelta': 100000, 'missDelta': 26918, 'missRatio': 0.26918}, '0': {'reqDelta': 0, 'missDelta': 0, 'missRatio': 0}, '400000': {'reqDelta': 100000, 'missDelta': 57264, 'missRatio': 0.57264}, '3500000': {'reqDelta': 100000, 'missDelta': 26974, 'missRatio': 0.26974}, '4900000': {'reqDelta': 100000, 'missDelta': 26581, 'missRatio': 0.26581}, '54600000': {'reqDelta': 100000, 'missDelta': 26941, 'missRatio': 0.26941}, '73100000': {'reqDelta': 100000, 'missDelta': 26744, 'missRatio': 0.26744}, '75300000': {'reqDelta': 100000, 'miss

In [66]:
optimal_lookup_dict[256]

0.190461575

In [56]:
target_df['directory'].unique()

array(['synth_static_202_bd515fcd-69db-4a69-9c68-b0450ffe9bb3',
       'synth_static_202_aac632b5-9fca-460c-a712-389b2885898b',
       'synth_static_202_cbe9e85d-2e33-40b2-a38c-8fca31f60a15',
       'synth_static_202_6a7168c3-7c94-4214-9aa1-095ceb5a9cfc',
       'synth_static_202_efa9a886-c372-4baa-9020-4f2a054476db',
       'synth_static_202_eca15395-9831-4541-9b43-6af9dec93834',
       'synth_static_202_38576bb3-9c4f-4083-8b20-c26c4b882524',
       'synth_static_202_c5337d9a-51c7-436e-9e07-3d65dc208420'],
      dtype=object)

In [65]:
target_df[['rebalanceStrategy', 'wakeUpRebalancerEveryXReqs', '_missRatio', "_rebalancerNumRebalancedSlabs", 'slab_cnt']]

,rebalanceStrategy,wakeUpRebalancerEveryXReqs,_missRatio,_rebalancerNumRebalancedSlabs,slab_cnt
40,disabled,5000,0.273442,0,256
50,marginal-hits,1000000,0.222395,76,256
97,disabled,5000,0.273442,0,256
344,marginal-hits,100000,0.190178,770,256
573,marginal-hits,10000,0.207165,6967,256
716,marginal-hits,500000,0.202443,153,256
749,marginal-hits,100000,0.190178,770,256
772,marginal-hits,1000,0.271491,56548,256


In [63]:
import ast
different_intervals_miss_ratios = []
for record in target_df.to_dict(orient='records'):
    miss_ratios = ast.literal_eval(record['_missRatios'])
    slab_cnt = record['slab_cnt']
    rebalanced_strategy = record['rebalanceStrategy']
    rebalanced_interval = record['wakeUpRebalancerEveryXReqs']
    for k, v in miss_ratios.items():
        request_id = int(k)
        miss_ratio = v['missRatio']
        different_intervals_miss_ratios.append({
            'rebalance_strategy': rebalanced_strategy if  rebalanced_strategy != 'marginal-hits' else f'marginal-hits-interval-{rebalanced_interval//1000}k',
            'slab_count': slab_cnt,
            'rebalance_interval': rebalanced_interval,
            'request_id': request_id,
            'miss_ratio': miss_ratio
        })
different_intervals_miss_ratios_df = pd.DataFrame(different_intervals_miss_ratios)

In [54]:
different_intervals_miss_ratios_df.rebalance_strategy.unique()

array(['disabled', 'marginal-hits-interval-1000k',
       'marginal-hits-interval-100k', 'marginal-hits-interval-10k',
       'marginal-hits-interval-500k', 'marginal-hits-interval-1k'],
      dtype=object)

In [64]:
different_intervals_miss_ratios_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_different_intervals_miss_ratios_256.csv", index=False)
different_intervals_miss_ratios_df

,rebalance_strategy,slab_count,rebalance_interval,request_id,miss_ratio
0,disabled,256,5000,51300000,0.26994
1,disabled,256,5000,46200000,0.26910
2,disabled,256,5000,40500000,0.26887
3,disabled,256,5000,33300000,0.26949
4,disabled,256,5000,31400000,0.26776
...,...,...,...,...,...
6395,marginal-hits-interval-1k,256,1000,62900000,0.26830
6396,marginal-hits-interval-1k,256,1000,63700000,0.26274
6397,marginal-hits-interval-1k,256,1000,47800000,0.26543
6398,marginal-hits-interval-1k,256,1000,42900000,0.26044


In [32]:
subdirs = [
    "synth_static_202_marginal-hits_256_1000",
    "synth_static_202_marginal-hits_256_10000",
    "synth_static_202_marginal-hits_256",
    "synth_static_202_marginal-hits_256_500000",
    "synth_static_202_marginal-hits_256_1000000"
]
"""
1. miss ratio over time -> analysis_df
2. distance to optimal allocation over time 

optimal: 89, 26, 97, 28, 16
initial: 29, 28, 72, 63, 64

"""
optimal = [89, 26, 97, 28, 16]
different_intervals_distance_to_optimal = []
import json
for subdir in subdirs:
    with open(os.path.join("outcome_detail", subdir, "out.json"), 'r') as f:
        data = json.load(f)
    with open(os.path.join("outcome_detail", subdir, "config.json"), 'r') as f:
        config = json.load(f)
    with open(os.path.join("outcome_detail", subdir, "stats.json"), 'r') as f:
        stats = json.load(f)
    
    slab_cnt = (config["cache_config"]["cacheSizeMB"] - 4) // 4
    rebalance_strategy = config["cache_config"]["rebalanceStrategy"]
    
    decisions = data['decision'] 
    current_allocation = [29, 28, 72, 53, 64]
    prev_distance_to_optimal_allocation = sum(
        abs(current_allocation[i] - optimal[i]) for i in range(len(optimal))
    ) // 2
    for decision in decisions:
        request_id, victim_id, receiver_id = decision['request_id'], decision['victim']['id'], decision['receiver']['id']
        interval = config["cache_config"]["wakeUpRebalancerEveryXReqs"]
        current_allocation[victim_id] -= 1
        current_allocation[receiver_id] += 1
        distance_to_optimal_allocation = sum(
            abs(current_allocation[i] - optimal[i]) for i in range(len(optimal))
        ) // 2
        different_intervals_distance_to_optimal.append({
            'rebalance_strategy': rebalance_strategy if rebalance_strategy != 'marginal-hits' else f"marginal-hits-interval-{interval//1000}k",
            'slab_count': slab_cnt,
            'request_id': request_id,
            'rebalance_interval': interval,
            'prev_distance_to_optimal_allocation': prev_distance_to_optimal_allocation,
            'distance_to_optimal_allocation': distance_to_optimal_allocation,
            'improvement': distance_to_optimal_allocation < prev_distance_to_optimal_allocation
        })
        prev_distance_to_optimal_allocation = distance_to_optimal_allocation
        
    

different_intervals_distance_to_optimal_df = pd.DataFrame(different_intervals_distance_to_optimal)



In [35]:
different_intervals_distance_to_optimal_df['rebalance_strategy'].unique()

array(['marginal-hits-interval-1k', 'marginal-hits-interval-10k',
       'marginal-hits-interval-100k', 'marginal-hits-interval-500k',
       'marginal-hits-interval-1000k'], dtype=object)

In [34]:
different_intervals_distance_to_optimal_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_256_different_intervals_distance_to_optimal.csv", index=False)

In [37]:
different_intervals_distance_to_optimal_df

,rebalance_strategy,slab_count,request_id,rebalance_interval,prev_distance_to_optimal_allocation,distance_to_optimal_allocation,improvement
0,marginal-hits-interval-1k,256,2811000,1000,80,79,True
1,marginal-hits-interval-1k,256,2812000,1000,79,78,True
2,marginal-hits-interval-1k,256,2813000,1000,78,79,False
3,marginal-hits-interval-1k,256,2814000,1000,79,80,False
4,marginal-hits-interval-1k,256,2815000,1000,80,79,True
...,...,...,...,...,...,...,...
64525,marginal-hits-interval-1000k,256,75000000,1000000,10,10,False
64526,marginal-hits-interval-1000k,256,76000000,1000000,10,10,False
64527,marginal-hits-interval-1000k,256,77000000,1000000,10,10,False
64528,marginal-hits-interval-1000k,256,78000000,1000000,10,10,False


In [ ]:
optimal_allocation_slabs_lookup[256]

{256: 89, 512: 26, 1024: 97, 2048: 28, 4096: 16}

In [27]:
"""
{'allSlabsAllocated': False,
 'request_id': 100000,
 'freeMemory': {'4': 0, '3': 0, '2': 0, '1': 0, '0': 0},
 'numSlabs': {'4': 8, '3': 7, '2': 5, '1': 2, '0': 2},
 'missEstimation': {'4': 12351, '3': 7637, '2': 540, '1': 5599, '0': 141},
 'hitsPerSlab': {'4': 1543, '3': 1091, '2': 108, '1': 2799, '0': 70},
 'evictions': {'4': 0, '3': 0, '2': 0, '1': 0, '0': 0},
 'hits': {'4': 12351, '3': 7637, '2': 540, '1': 5599, '0': 141},
 'marginalHits': {'4': 690, '3': 834, '2': 178, '1': 1930, '0': 110},
 'tailAge': {'4': 756, '3': 756, '2': 756, '1': 756, '0': 756}}
"""
import json

detail_subdirs = list_subdirs("outcome_detail")
stats_detail = []
for subdir in detail_subdirs:
    with open(os.path.join("outcome_detail", subdir, "out.json"), 'r') as f:
        data = json.load(f)
    with open(os.path.join("outcome_detail", subdir, "config.json"), 'r') as f:
        config = json.load(f)
    with open(os.path.join("outcome_detail", subdir, "stats.json"), 'r') as f:
        stats = json.load(f)
    
    slab_cnt = (config["cache_config"]["cacheSizeMB"] - 4) // 4
    rebalance_strategy = config["cache_config"]["rebalanceStrategy"]
    for snap in data['snapshots']:
        for k, v in snap['numSlabs'].items():
            stats_detail.append({
                'rebalance_strategy': rebalance_strategy,
                'slab_count': slab_cnt,
                'request_id': snap['request_id'],
                'value': v,
                'key': 'num_slabs',
                'class_id': int(k) 
            })
        
        
        for k, v in snap['freeMemory'].items():
            stats_detail.append({
                'rebalance_strategy': rebalance_strategy,
                'slab_count': slab_cnt,
                'request_id': snap['request_id'],
                'value': v // (4 * 1024 * 1024),
                'key': 'free_slabs',
                'class_id': int(k) 
            })
        for k, v in snap['hitsPerSlab'].items():
            stats_detail.append({
                'rebalance_strategy': rebalance_strategy,
                'slab_count': slab_cnt,
                'request_id': snap['request_id'],
                'value': v,
                'key': 'hits_per_slab',
                'class_id': int(k) 
            })
        for k, v in snap['marginalHits'].items():
            stats_detail.append({
                'rebalance_strategy': rebalance_strategy,
                'slab_count': slab_cnt,
                'request_id': snap['request_id'],
                'key': 'marginal_hits',
                'value': v,
                'class_id': int(k) 
            })
        for k, v in snap['tailAge'].items():
            stats_detail.append({
                'rebalance_strategy': rebalance_strategy,
                'slab_count': slab_cnt,
                'request_id': snap['request_id'],
                'key': 'tail_age',
                'value': v,
                'class_id': int(k) 
            })
stats_detail_df = pd.DataFrame(stats_detail)
    

In [28]:
stats_detail_df.to_csv(f"/mydata/hongshu/thesis-playground/thesis-plotting/scripts/compare/{trace_name}_stats_detail.csv", index=False)

In [19]:
distance_to_optimal_df[distance_to_optimal_df['rebalance_strategy'] == 'marginal-hits']

,rebalance_strategy,slab_count,request_id,distance_to_optimal_allocation
2397,marginal-hits,256,100000,116
2398,marginal-hits,256,200000,107
2399,marginal-hits,256,300000,101
2400,marginal-hits,256,400000,98
2401,marginal-hits,256,500000,95
...,...,...,...,...
9583,marginal-hits,64,79500000,2
9584,marginal-hits,64,79600000,3
9585,marginal-hits,64,79700000,3
9586,marginal-hits,64,79800000,4


In [31]:
pd.DataFrame(all_allocated_results).sort_values(by='all_slabs_allocated_req_id')

,rebalance_strategy,all_slabs_allocated_req_id,total_slab_cnt
0,free-mem,200000,32
7,hits,200000,32
6,marginal-hits,200000,32
9,disabled,200000,32
16,tail-age,200000,32
15,tail-age,400000,64
10,free-mem,400000,64
11,marginal-hits,400000,64
19,hits,400000,64
5,disabled,400000,64


In [32]:
analysis_df

,directory,allocator,lruRefreshSec,cacheSizeMB,moveOnSlabRelease,rebalanceStrategy,poolRebalanceIntervalSec,wakeUpRebalancerEveryXReqs,allocSizes,trace_name,...,_poolFragmentationSize,_intervalChangeEvents,_rebalanceIntervals,_anomalyReqIds,tailSlabCnt,mhMovingAverageParam,rebalanceDiffRatio,_missRatio,slab_cnt,class_slabs
0,synth_static_202_bd515fcd-69db-4a69-9c68-b0450...,LRU2Q,0,1028,False,disabled,1,5000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,2387968,{'0': 'init'},{'0': 5000},[],NaN,NaN,NaN,0.273442,256,"{256: 29, 512: 28, 1024: 72, 2048: 63, 4096: 64}"
1,synth_static_202_b8e06995-0baa-40cd-bc19-1e08e...,LRU2Q,0,260,False,marginal-hits,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,813056,{'0': 'init'},{'0': 100000},[],1.0,0.3,NaN,0.554192,64,"{256: 12, 512: 14, 1024: 13, 2048: 16, 4096: 9}"
2,synth_static_202_784eb810-c1b4-4295-bad1-ea8e3...,LRU2Q,0,132,False,free-mem,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,231424,{'0': 'init'},{'0': 100000},[],NaN,NaN,0.25,0.640658,32,"{256: 2, 512: 3, 1024: 7, 2048: 9, 4096: 11}"
3,synth_static_202_fabd1377-206b-4eee-a585-2f917...,LRU2Q,0,1028,False,hits,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,2363392,{'0': 'init'},{'0': 100000},[],NaN,NaN,0.10,0.349339,256,"{256: 18, 512: 72, 1024: 17, 2048: 73, 4096: 76}"
4,synth_static_202_a444a4fe-1bbe-4340-bceb-f3f1e...,LRU2Q,0,260,False,hits,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,479232,{'0': 'init'},{'0': 100000},[],NaN,NaN,0.10,0.565739,64,"{256: 1, 512: 19, 1024: 1, 2048: 19, 4096: 24}"
5,synth_static_202_70743971-929d-4a33-9be1-08ab2...,LRU2Q,0,260,False,tail-age,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,503808,{'0': 'init'},{'0': 100000},[],NaN,NaN,0.25,0.564820,64,"{256: 5, 512: 6, 1024: 16, 2048: 17, 4096: 20}"
6,synth_static_202_9b4e6e81-a637-40b2-a9ce-38844...,LRU2Q,0,516,False,free-mem,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,1060864,{'0': 'init'},{'0': 100000},[],NaN,NaN,0.25,0.452097,128,"{256: 11, 512: 12, 1024: 36, 2048: 33, 4096: 36}"
7,synth_static_202_6a7168c3-7c94-4214-9aa1-095ce...,LRU2Q,0,1028,False,marginal-hits,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,4487086,{'0': 'init'},{'0': 100000},[],1.0,0.3,NaN,0.190178,256,"{256: 99, 512: 23, 1024: 98, 2048: 23, 4096: 13}"
8,synth_static_202_b81d2a15-b8e7-4cf6-8b6c-e9a4f...,LRU2Q,0,132,False,disabled,1,5000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,231424,{'0': 'init'},{'0': 5000},[],NaN,NaN,NaN,0.640658,32,"{256: 2, 512: 3, 1024: 7, 2048: 9, 4096: 11}"
9,synth_static_202_93693825-dcea-4b55-a766-32af6...,LRU2Q,0,516,False,marginal-hits,1,100000,"[256, 512, 1024, 2048, 4096]",synth_static_202,...,2080492,{'0': 'init'},{'0': 100000},[],1.0,0.3,NaN,0.419301,128,"{256: 41, 512: 18, 1024: 44, 2048: 16, 4096: 9}"


In [ ]:
"""
request_id, rebalance_strategy, slab_count, mi
"""